# Tugas 4 No. 1 — Preprocessing Teks
**Topik:** Tokenisasi, Stopwords Removal, dan Stemming (Bahasa Indonesia)

**Nama:** Nurul Fadhyah
**NIM:** *(isi NIM Anda di sini)*
**Mata Kuliah:** Temu Kembali Informasi

## Studi Kasus
Bagian pengembangan sistem pencarian dokumen internal Fakultas Teknik UNM memiliki kumpulan berita
dan pengumuman resmi yang masih "kotor" (mengandung tanda baca, huruf kapital, stopwords, dan kata
berimbuhan). Notebook ini membangun pipeline `preprocess_text()` untuk membersihkan data tersebut
agar siap digunakan pada sistem temu kembali informasi (Information Retrieval).

**Pipeline:** Case Folding → Cleaning → Tokenisasi → Stopwords Removal (Sastrawi) → Stemming (Sastrawi)

## 1. Import Library

In [ ]:
import re
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi stopword remover dan stemmer dari Sastrawi
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

## 2. Fungsi `preprocess_text(text)`

Fungsi ini menjalankan seluruh tahapan preprocessing secara berurutan dan mengembalikan
daftar token akhir. Parameter `return_steps=True` opsional digunakan untuk menampilkan
hasil pada setiap tahap (dipakai pada bagian perbandingan sebelum/sesudah).

In [ ]:
def preprocess_text(text, return_steps=False):
    """
    Pipeline preprocessing teks Bahasa Indonesia:
    1. Case folding
    2. Cleaning (hapus angka, tanda baca, karakter khusus)
    3. Tokenisasi
    4. Stopwords removal (Sastrawi)
    5. Stemming (Sastrawi)

    Return:
        list token hasil preprocessing (atau tuple (tokens, steps) jika return_steps=True)
    """
    steps = {}

    # 1. Case folding
    case_folded = text.lower()
    steps['1_case_folding'] = case_folded

    # 2. Cleaning: hapus angka, tanda baca, dan karakter khusus, sisakan huruf & spasi
    cleaned = re.sub(r'[0-9]+', ' ', case_folded)
    cleaned = re.sub(r'[^a-z\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    steps['2_cleaning'] = cleaned

    # 3. Tokenisasi
    tokens = cleaned.split()
    steps['3_tokenisasi'] = tokens

    # 4. Stopwords removal (Sastrawi bekerja pada level string)
    no_stopwords_str = stopword_remover.remove(' '.join(tokens))
    tokens_no_stopwords = no_stopwords_str.split()
    steps['4_stopwords_removal'] = tokens_no_stopwords

    # 5. Stemming (Sastrawi bekerja pada level string)
    stemmed_str = stemmer.stem(' '.join(tokens_no_stopwords))
    stemmed_tokens = stemmed_str.split()
    steps['5_stemming'] = stemmed_tokens

    if return_steps:
        return stemmed_tokens, steps
    return stemmed_tokens

## 3. Dataset

Lima dokumen contoh (berita/pengumuman kampus) dengan tema teknologi, komputer, dan pendidikan.
Dokumen sengaja dibuat "kotor" (huruf kapital, tanda baca, angka) agar dampak preprocessing terlihat jelas.

In [ ]:
documents = [
    "Fakultas Teknik Universitas Negeri Makassar mengembangkan sebuah sistem pencarian dokumen "
    "internal yang memanfaatkan teknologi kecerdasan buatan sejak tahun 2026.",

    "Mahasiswa Program Studi Teknik Komputer diwajibkan untuk mengikuti pelatihan pemrograman "
    "Python serta analisis data pada semester ini!",

    "Pengumuman: Rapat koordinasi dosen Jurusan Teknik Informatika dan Komputer akan dilaksanakan "
    "pada tanggal 25 September 2026 di ruang rapat utama kampus.",

    "Perkembangan teknologi jaringan komputer generasi ke-5 (5G) membuka peluang baru bagi riset "
    "Internet of Things di lingkungan kampus.",

    "Sistem temu-kembali informasi atau information retrieval menjadi salah satu topik penting "
    "dalam mata kuliah Metode Penelitian tahun ini.",
]

for i, doc in enumerate(documents, 1):
    print(f"Dokumen {i}: {doc}\n")

## 4. Terapkan `preprocess_text()` pada Kelima Dokumen

In [ ]:
results = []          # token akhir tiap dokumen
all_steps = []        # detail setiap tahap tiap dokumen

for doc in documents:
    tokens, steps = preprocess_text(doc, return_steps=True)
    results.append(tokens)
    all_steps.append(steps)

print("Selesai memproses", len(documents), "dokumen.")

## 5. Perbandingan Sebelum vs Sesudah Preprocessing

Ditampilkan secara lengkap (seluruh tahap) untuk **Dokumen 1** dan **Dokumen 2**,
lalu ringkasan hasil akhir untuk seluruh dokumen.

In [ ]:
def print_full_comparison(doc_index):
    doc = documents[doc_index]
    steps = all_steps[doc_index]
    print(f"===== DOKUMEN {doc_index + 1} =====")
    print("Teks asli           :", doc)
    print("1. Case folding     :", steps['1_case_folding'])
    print("2. Cleaning         :", steps['2_cleaning'])
    print("3. Tokenisasi       :", steps['3_tokenisasi'])
    print("4. Stopwords removal:", steps['4_stopwords_removal'])
    print("5. Stemming (final) :", steps['5_stemming'])
    print()

print_full_comparison(0)
print_full_comparison(1)

In [ ]:
print("===== RINGKASAN SELURUH DOKUMEN (SEBELUM -> SESUDAH) =====\n")
for i, doc in enumerate(documents):
    print(f"Dokumen {i+1}")
    print("  Sebelum :", doc)
    print("  Sesudah :", ' '.join(results[i]))
    print()

## 6. Statistik Jumlah Token

- **Jumlah token sebelum preprocessing**: hasil `split()` sederhana pada teks mentah (tanpa dibersihkan).
- **Jumlah token sesudah preprocessing**: panjang token akhir hasil `preprocess_text()`.
- **Persentase pengurangan**: `(sebelum - sesudah) / sebelum * 100`.

In [ ]:
summary_rows = []
for i, doc in enumerate(documents):
    n_before = len(doc.split())
    n_after = len(results[i])
    reduction_pct = (n_before - n_after) / n_before * 100
    summary_rows.append({
        "Dokumen": f"Dokumen {i+1}",
        "Token Sebelum": n_before,
        "Token Sesudah": n_after,
        "Pengurangan (%)": round(reduction_pct, 2)
    })

summary_df = pd.DataFrame(summary_rows)

total_before = summary_df["Token Sebelum"].sum()
total_after = summary_df["Token Sesudah"].sum()
total_reduction = (total_before - total_after) / total_before * 100

summary_df.loc[len(summary_df)] = ["TOTAL", total_before, total_after, round(total_reduction, 2)]

summary_df

## 7. Analisis Singkat

Preprocessing memberikan dampak positif yang signifikan terhadap kualitas data untuk keperluan sistem
*Information Retrieval*. Proses *case folding* dan *cleaning* menyeragamkan format teks sehingga kata
yang sama tidak dianggap berbeda hanya karena huruf kapital, tanda baca, atau angka yang menyertainya.
*Stopwords removal* membuang kata-kata umum (seperti "dan", "yang", "untuk") yang sering muncul di
hampir semua dokumen namun tidak memiliki daya pembeda (*discriminative power*) dalam proses pencarian,
sehingga indeks yang terbentuk menjadi lebih ringkas dan relevan. Sementara itu, *stemming* menyatukan
kata-kata berimbuhan ke bentuk dasarnya (misalnya "mengembangkan" dan "dikembangkan" menjadi
"kembang"), sehingga dokumen yang menggunakan variasi kata yang berbeda namun bermakna sama tetap dapat
ditemukan saat query dicocokkan. Secara keseluruhan, pipeline ini mengurangi jumlah token secara
signifikan (lihat tabel di atas) sekaligus meningkatkan kualitas representasi teks, yang pada akhirnya
akan mempercepat proses pengindeksan dan meningkatkan akurasi/relevansi hasil pencarian pada sistem
temu kembali informasi.